# Pipeline V8 — Fase 3: Rotulação/Re-rotulação Databricks (definitivo)

**Spec**: `specs/004-melhoria-geracao-dados-cnn/` (T-A3-007). **Execução: Databricks.**

Calcula `melhor_jogada`, `score_melhor_jogada` e `depth_melhor_jogada` para qualquer
lote de NPZs com schema v2-a3 (12 canais + 3 escalares de cadeias já presentes).

**Uso primário**: rotular NPZs recém-gerados pelo pipeline V7 DAC (Fase 1) após
enriquecimento pela Fase 2. Também serve para re-rotular datasets existentes com
profundidade maior.

### Estratégia de profundidade adaptativa

```
depth_estado = DEPTH_ADAPTATIVO  se  arestas_livres > 11  E  prof_min > 11
             = DEPTH_PADRAO       caso contrário

onde:  arestas_livres = 31 − qtd_tracos
       prof_min       = total_caixas_cadeias_longas + 2 × qtd_cadeias_longas
```

O Minimax para naturalmente quando o jogo termina; `DEPTH_ADAPTATIVO=20` resolve
qualquer estado com ≤ 19 arestas livres (máximo observado no dataset).

### Otimizações (baseadas em Geracao_Amostras_v7_adaptativo_Fase_2_HighPerf.ipynb)

1. **Databricks Serverless**: usa `mapInPandas` — sem RDDs, evita `JVM_ATTRIBUTE_NOT_SUPPORTED`.
2. **Profundidade por estado**: coluna `depth` enviada a cada worker — estados com cadeias longas
   recebem `DEPTH_ADAPTATIVO`, demais recebem `DEPTH_PADRAO`.
3. **Checkpointing (lotes de `LOTE_ARQUIVOS`)**: em caso de queda do cluster, retoma do último lote.
4. **Gravação atômica**: `.tmp.npz` + `os.replace()` — nunca corrompe originais.
5. **Preservação total do schema v2-a3**: canais, nomes_canais e 3 escalares são mantidos.

> **⚠️ ALERTA DE MANUTENÇÃO: BUGS ALGÓRITMICOS NO BITBOARD (Maio/2026)**
>
> O Minimax via Bitboard neste notebook possui dois bugs que foram corrigidos e
> **NÃO DEVEM ser revertidos ou esquecidos** ao adaptar este código:
>
> **Bug 1 — Caixas Pré-Fechadas (Falsos Positivos):** ao contar caixas completadas
> por uma jogada, é OBRIGATÓRIO verificar que a caixa *não estava fechada antes*:
> ```python
> # CORRETO:
> closed = sum(1 for bm in edge_boxes[i] if new_e & bm == bm and edges & bm != bm)
> # ERRADO (conta caixas já fechadas antes da jogada):
> closed = sum(1 for bm in edge_boxes[i] if new_e & bm == bm)
> ```
>
> **Bug 2 — Offsets na Poda Alpha-Beta Incremental:** este Bitboard retorna scores
> *incrementais* (não absolutos). Ao chamar a subárvore após capturar `closed` caixas,
> os limites alpha/beta DEVEM ser ajustados com offset de `−closed`:
> ```python
> # CORRETO (maximizando, capturou `closed` caixas — mesma vez):
> val = closed + solve_minimax_bitboard(new_e, depth-1, alpha - closed, beta - closed, True, tt)
> # ERRADO (repassa alpha/beta puro — causa poda prematura e escolhe jogadas perdedoras):
> val = closed + solve_minimax_bitboard(new_e, depth-1, alpha, beta, True, tt)
> ```

In [0]:
import os
import time
import random
import numpy as np
import pandas as pd
from pathlib import Path
from pyspark.sql import functions as F

# ── Configurações ──────────────────────────────────────────────────────────────
# Path do diretório NPZ no DBFS ou montagem S3/Azure.
INPUT_DIR = Path('/Workspace/Users/c092820@corp.caixa.gov.br/CNN/profundidade_minimax_11_v7_adaptativo')

# Profundidade padrão para estados sem cadeias longas problemáticas.
DEPTH_PADRAO = 11

# Profundidade usada quando arestas_livres > 11 E prof_min > 11.
# Minimax para naturalmente — p=20 resolve qualquer estado com <= 19 arestas livres.
DEPTH_ADAPTATIVO = 20

# Número de NPZs por lote (checkpoint). ~4-8 NPZs = ~20-40k estados por lote.
LOTE_ARQUIVOS = 16
NUM_PARTICOES = 360

# Se True: re-rotula TODOS os estados, inclusive os já rotulados.
# Se False: pula estados onde depth_melhor_jogada[i] >= depth_necessaria[i].
FORCAR_REGRAVAR = False

print(f'INPUT_DIR        = {INPUT_DIR}')
print(f'DEPTH_PADRAO     = {DEPTH_PADRAO}')
print(f'DEPTH_ADAPTATIVO = {DEPTH_ADAPTATIVO}')
print(f'LOTE_ARQUIVOS    = {LOTE_ARQUIVOS}')
print(f'FORCAR_REGRAVAR  = {FORCAR_REGRAVAR}')

INPUT_DIR        = /Workspace/Users/c092820@corp.caixa.gov.br/CNN/profundidade_minimax_11_v7_adaptativo
DEPTH_PADRAO     = 11
DEPTH_ADAPTATIVO = 20
LOTE_ARQUIVOS    = 16
FORCAR_REGRAVAR  = False


In [0]:
# ── Diagnóstico: contar estados por depth necessária ───────────────────────────
# Confirmar distribuição antes de rodar o processamento.

SUFIXOS_SIMETRIA = ('_refH', '_refV', '_r180')
todos_npzs = sorted(INPUT_DIR.glob('dataset_pequeno_*.npz'))
npzs_originais = [
    p for p in todos_npzs
    if not any(p.stem.endswith(s) for s in SUFIXOS_SIMETRIA)
    and '.tmp.' not in p.name
]

print(f'NPZs originais: {len(npzs_originais)}')

n_padrao = 0
n_adaptativo = 0
n_pendentes_padrao = 0
n_pendentes_adaptativo = 0

for path in npzs_originais:
    with np.load(path, allow_pickle=False) as d:
        qtd_tracos  = d['qtd_tracos'].astype(np.int32)
        qtd_cad     = d['qtd_cadeias_longas'].astype(np.int32)
        total_cad   = d['total_caixas_cadeias_longas'].astype(np.int32)
        depth_atual = d['depth_melhor_jogada'].astype(np.int32) if 'depth_melhor_jogada' in d.files else np.zeros(len(qtd_tracos), dtype=np.int32)
        rotulados   = (d['melhor_jogada'] != '') if 'melhor_jogada' in d.files else np.zeros(len(qtd_tracos), dtype=bool)

    arestas_livres = 31 - qtd_tracos
    prof_min = total_cad + 2 * qtd_cad
    precisa_adaptativo = (arestas_livres > 11) & (prof_min > 11)
    depth_necessaria = np.where(precisa_adaptativo, DEPTH_ADAPTATIVO, DEPTH_PADRAO)

    n_padrao      += int((~precisa_adaptativo).sum())
    n_adaptativo  += int(precisa_adaptativo.sum())

    if FORCAR_REGRAVAR:
        pendente = np.ones(len(qtd_tracos), dtype=bool)
    else:
        pendente = ~rotulados | (depth_atual < depth_necessaria)

    n_pendentes_padrao      += int((pendente & ~precisa_adaptativo).sum())
    n_pendentes_adaptativo  += int((pendente & precisa_adaptativo).sum())

total_estados = n_padrao + n_adaptativo
total_pendentes = n_pendentes_padrao + n_pendentes_adaptativo
print(f'\nTotal de estados       : {total_estados:,}')
print(f'  Depth {DEPTH_PADRAO} (padrão)   : {n_padrao:,}')
print(f'  Depth {DEPTH_ADAPTATIVO} (adaptativo) : {n_adaptativo:,}')
print(f'\nPendentes (a rotular)  : {total_pendentes:,} ({100*total_pendentes/total_estados:.2f}%)')
print(f'  com depth {DEPTH_PADRAO}         : {n_pendentes_padrao:,}')
print(f'  com depth {DEPTH_ADAPTATIVO}         : {n_pendentes_adaptativo:,}')

NPZs originais: 152

Total de estados       : 758,640
  Depth 11 (padrão)   : 747,098
  Depth 20 (adaptativo) : 11,542

Pendentes (a rotular)  : 418,640 (55.18%)
  com depth 11         : 412,308
  com depth 20         : 6,332


In [0]:
# ── Worker PySpark: Minimax Bitboard com profundidade por estado ───────────────
# Toda a lógica está inline para evitar erros de serialização (Pickle) no
# Databricks Serverless. Ambos os bug fixes do Bitboard estão aplicados.

def processar_lote_pandas(iterator):
    """
    Função enviada aos Workers PySpark via mapInPandas.
    Recebe DataFrame com colunas: estado_bytes (BINARY), depth (INT).
    Retorna: estado_bytes, depth, melhor_jogada (STRING), scores (ARRAY<FLOAT>).
    """
    import random
    import numpy as np
    import pandas as pd

    # ── Reconstrói tabelas de Bitboard localmente no Worker ──────────────────
    h, w = 9, 7
    edge_to_bit = {}
    bit_to_label = {}
    bit_idx = 0
    for r in range(h):
        for c in range(w):
            if (r % 2 == 0 and c % 2 == 1) or (r % 2 == 1 and c % 2 == 0):
                edge_to_bit[(r, c)] = bit_idx
                bit_to_label[bit_idx] = f'H_{r}_{c}' if r % 2 == 0 else f'V_{r}_{c}'
                bit_idx += 1

    n_edges = bit_idx
    all_mask = (1 << n_edges) - 1

    box_masks = []
    for r in range(1, h, 2):
        for c in range(1, w, 2):
            mask = (1 << edge_to_bit[(r-1, c)]) | (1 << edge_to_bit[(r+1, c)]) | \
                   (1 << edge_to_bit[(r, c-1)]) | (1 << edge_to_bit[(r, c+1)])
            box_masks.append(mask)

    edge_boxes = [tuple(bm for bm in box_masks if bm & (1 << b)) for b in range(n_edges)]

    # ── Minimax Bitboard ──────────────────────────────────────────────────────
    # Fix 1: `edges & bm != bm` — exclui caixas pré-fechadas.
    # Fix 2: offset alpha/beta por `closed` — poda incremental correta.
    def solve_minimax_bitboard(edges, depth, alpha, beta, maximizing, tt):
        if depth == 0 or edges == all_mask:
            return 0

        tt_key = (edges, depth, maximizing)
        if tt_key in tt:
            flag, val = tt[tt_key]
            if flag == 0: return val
            if flag == 1 and val >= beta: return val
            if flag == 2 and val <= alpha: return val

        moves = []
        for i in range(n_edges):
            if not (edges & (1 << i)):
                ne = edges | (1 << i)
                # Fix 1: conta apenas caixas que foram FECHADAS AGORA
                closed = sum(1 for bm in edge_boxes[i] if ne & bm == bm and edges & bm != bm)
                moves.append((i, closed))
        moves.sort(key=lambda x: x[1], reverse=True)

        orig_alpha = alpha
        orig_beta = beta
        best_val = -10000 if maximizing else 10000

        for bit, closed in moves:
            new_e = edges | (1 << bit)
            if maximizing:
                if closed > 0:
                    # Fix 2: offset alpha/beta incremental — mesma vez (capturou e joga de novo)
                    val = closed + solve_minimax_bitboard(new_e, depth-1, alpha - closed, beta - closed, True, tt)
                else:
                    val = solve_minimax_bitboard(new_e, depth-1, alpha, beta, False, tt)
                best_val = max(best_val, val)
                alpha = max(alpha, best_val)
            else:
                if closed > 0:
                    # Fix 2: adversário capturou — offset invertido
                    val = -closed + solve_minimax_bitboard(new_e, depth-1, alpha + closed, beta + closed, False, tt)
                else:
                    val = solve_minimax_bitboard(new_e, depth-1, alpha, beta, True, tt)
                best_val = min(best_val, val)
                beta = min(beta, best_val)
            if beta <= alpha:
                break

        # TT com flags EXACT / LOWERBOUND / UPPERBOUND
        if maximizing:
            flag = 2 if best_val <= orig_alpha else (1 if best_val >= beta else 0)
        else:
            flag = 1 if best_val >= orig_beta else (2 if best_val <= alpha else 0)
        tt[tt_key] = (flag, best_val)
        return best_val

    # ── Processamento dos DataFrames recebidos do Spark ───────────────────────
    for pdf in iterator:
        resultados = []
        for row in pdf.itertuples(index=False):
            estado_bytes = bytes(row.estado_bytes)
            depth = int(row.depth)

            mat = np.frombuffer(estado_bytes, dtype=np.int8).reshape(9, 7)
            edges = 0
            idx = 0
            for r in range(9):
                for c in range(7):
                    if (r % 2 == 0 and c % 2 == 1) or (r % 2 == 1 and c % 2 == 0):
                        if mat[r, c] == 9:
                            edges |= (1 << idx)
                        idx += 1

            # TT reiniciada por estado (não compartilhar entre tabuleiros diferentes)
            tt = {}
            scores = np.full(31, -1_000_000_000.0, dtype=np.float32)

            for i in range(31):
                if not (edges & (1 << i)):
                    new_e = edges | (1 << i)
                    closed = sum(1 for bm in edge_boxes[i] if new_e & bm == bm and edges & bm != bm)
                    if closed > 0:
                        res = closed + solve_minimax_bitboard(new_e, depth-1, -10001, 10001, True, tt)
                    else:
                        res = solve_minimax_bitboard(new_e, depth-1, -10001, 10001, False, tt)
                    scores[i] = float(res)

            validos = [i for i, s in enumerate(scores) if s > -1e8]
            if not validos:
                melhor_rotulo = ''
            else:
                max_s = max(scores[i] for i in validos)
                best_idx = random.choice([i for i in validos if scores[i] == max_s])
                melhor_rotulo = bit_to_label[best_idx]

            resultados.append({
                'estado_bytes': estado_bytes,
                'depth': depth,
                'melhor_jogada': melhor_rotulo,
                'scores': scores.tolist(),
            })
        yield pd.DataFrame(resultados)

print('Worker processar_lote_pandas() definido.')

Worker processar_lote_pandas() definido.


In [0]:
# ── Orquestração principal com checkpointing e gravação atômica ───────────────

spark.conf.set('spark.databricks.execution.timeout', 7200)

schema_spark = 'estado_bytes BINARY, depth INT, melhor_jogada STRING, scores ARRAY<FLOAT>'

# ── Monta a fila de NPZs pendentes ────────────────────────────────────────────
pendentes = []
print(f'Verificando {len(npzs_originais)} arquivos para montar a fila...')

for path in npzs_originais:
    if '.tmp.' in path.name:
        continue
    with np.load(path, allow_pickle=False) as d:
        qtd_tracos  = d['qtd_tracos'].astype(np.int32)
        qtd_cad     = d['qtd_cadeias_longas'].astype(np.int32)
        total_cad   = d['total_caixas_cadeias_longas'].astype(np.int32)
        depth_atual = d['depth_melhor_jogada'].astype(np.int32) if 'depth_melhor_jogada' in d.files else np.zeros(len(qtd_tracos), dtype=np.int32)
        rotulados   = d['melhor_jogada'] != '' if 'melhor_jogada' in d.files else np.zeros(len(qtd_tracos), dtype=bool)

    arestas_livres = 31 - qtd_tracos
    prof_min = total_cad + 2 * qtd_cad
    precisa_adaptativo = (arestas_livres > 11) & (prof_min > 11)
    depth_necessaria = np.where(precisa_adaptativo, DEPTH_ADAPTATIVO, DEPTH_PADRAO)

    if FORCAR_REGRAVAR:
        tem_pendente = True
    else:
        tem_pendente = bool((~rotulados | (depth_atual < depth_necessaria)).any())

    if tem_pendente:
        pendentes.append(path)

print(f'Arquivos já completos : {len(npzs_originais) - len(pendentes)}')
print(f'Arquivos na fila      : {len(pendentes)}')

Verificando 152 arquivos para montar a fila...
Arquivos já completos : 68
Arquivos na fila      : 84


In [0]:
# ── Loop de processamento em lotes ────────────────────────────────────────────

for i in range(0, len(pendentes), LOTE_ARQUIVOS):
    lote_paths = pendentes[i : i + LOTE_ARQUIVOS]

    # Extrai pares únicos (estado_bytes, depth_necessaria) do lote.
    # Usar (bytes, depth) como chave evita calcular duas vezes o mesmo estado
    # com a mesma profundidade, mas permite re-calcular com profundidade diferente.
    estados_para_calcular = {}  # (bytes, depth) -> depth  (para deduplicação)

    for path in lote_paths:
        with np.load(path, allow_pickle=False) as d:
            qtd_tracos  = d['qtd_tracos'].astype(np.int32)
            qtd_cad     = d['qtd_cadeias_longas'].astype(np.int32)
            total_cad   = d['total_caixas_cadeias_longas'].astype(np.int32)
            depth_atual = d['depth_melhor_jogada'].astype(np.int32) if 'depth_melhor_jogada' in d.files else np.zeros(len(qtd_tracos), dtype=np.int32)
            rotulados   = d['melhor_jogada'] != '' if 'melhor_jogada' in d.files else np.zeros(len(qtd_tracos), dtype=bool)
            estados_arr = d['estados']

        arestas_livres = 31 - qtd_tracos
        prof_min = total_cad + 2 * qtd_cad
        precisa_adaptativo = (arestas_livres > 11) & (prof_min > 11)
        depth_necessaria = np.where(precisa_adaptativo, DEPTH_ADAPTATIVO, DEPTH_PADRAO)

        for j, estado in enumerate(estados_arr):
            if not FORCAR_REGRAVAR and rotulados[j] and depth_atual[j] >= depth_necessaria[j]:
                continue
            chave = (estado.tobytes(), int(depth_necessaria[j]))
            estados_para_calcular[chave] = int(depth_necessaria[j])

    print(f'--- Lote {i // LOTE_ARQUIVOS + 1}/{(len(pendentes) + LOTE_ARQUIVOS - 1) // LOTE_ARQUIVOS} ({len(lote_paths)} NPZs) ---')
    print(f'    Estados únicos a calcular: {len(estados_para_calcular)}')

    if not estados_para_calcular:
        print('    Nenhum pendente neste lote. Pulando.')
        continue

    t_lote = time.time()

    # Envia para o cluster Spark com coluna de depth por estado.
    pdf_in = pd.DataFrame([
        {'estado_bytes': estado_b, 'depth': depth_val}
        for (estado_b, _), depth_val in estados_para_calcular.items()
    ])
    df_spark = spark.createDataFrame(pdf_in)

    #conf_particoes = spark.conf.get('spark.sql.shuffle.partitions', '200')
    #try:
    #    num_particoes = int(conf_particoes)
    #except ValueError:
    #    num_particoes = 200
    #df_spark = df_spark.repartition(num_particoes)
    # Embaralhamos os dados nas partições para que cada partição receba dados embaralhados (p=11 e p=20)
    df_spark = df_spark.orderBy(F.rand()).repartition(NUM_PARTICOES)
    print( f'    Partições: {NUM_PARTICOES} (~{len(estados_para_calcular) // NUM_PARTICOES} estados/partição)' )

    df_result = df_spark.mapInPandas(processar_lote_pandas, schema=schema_spark)
    resultados_lote = df_result.collect()

    # Cache local: (estado_bytes, depth) -> (melhor_jogada, scores)
    cache = {
        (bytes(row.estado_bytes), int(row.depth)): (row.melhor_jogada, np.array(row.scores, dtype=np.float32))
        for row in resultados_lote
    }

    print(f'    Spark: {len(cache)} resultados em {time.time() - t_lote:.1f}s. Gravando...')

    # Gravação atômica: preserva TODO o schema v2-a3.
    for path in lote_paths:
        with np.load(path, allow_pickle=False) as d:
            dados = dict(d)

        estados_arr = dados['estados']
        N = len(estados_arr)
        qtd_tracos  = dados['qtd_tracos'].astype(np.int32)
        qtd_cad     = dados['qtd_cadeias_longas'].astype(np.int32)
        total_cad   = dados['total_caixas_cadeias_longas'].astype(np.int32)

        arestas_livres = 31 - qtd_tracos
        prof_min = total_cad + 2 * qtd_cad
        precisa_adaptativo = (arestas_livres > 11) & (prof_min > 11)
        depth_necessaria = np.where(precisa_adaptativo, DEPTH_ADAPTATIVO, DEPTH_PADRAO)

        depth_atual = dados['depth_melhor_jogada'].astype(np.int32) if 'depth_melhor_jogada' in dados else np.zeros(N, dtype=np.int32)
        rotulados   = dados['melhor_jogada'] != '' if 'melhor_jogada' in dados else np.zeros(N, dtype=bool)

        mj_arr  = np.array(list(dados.get('melhor_jogada', [''] * N)), dtype='<U5')
        smj_arr = dados.get('score_melhor_jogada', np.full((N, 31), -1e9, dtype=np.float32)).copy()
        dm_arr  = depth_atual.copy().astype(np.int8)

        for j, estado in enumerate(estados_arr):
            if not FORCAR_REGRAVAR and rotulados[j] and depth_atual[j] >= depth_necessaria[j]:
                continue
            chave = (estado.tobytes(), int(depth_necessaria[j]))
            if chave in cache:
                mj_arr[j], smj_arr[j] = cache[chave]
                dm_arr[j] = np.int8(depth_necessaria[j])

        dados['melhor_jogada']        = mj_arr
        dados['score_melhor_jogada']  = smj_arr
        dados['depth_melhor_jogada']  = dm_arr

        tmp_path = path.with_name(path.stem + '.tmp.npz')
        np.savez_compressed(tmp_path, **dados)
        os.replace(tmp_path, path)
        print(f'    -> {path.name} gravado.')

print('\nTodo o processamento finalizado!')

--- Lote 1/6 (16 NPZs) ---
    Estados únicos a calcular: 62285
    Partições: 360 (~173 estados/partição)
    Spark: 62285 resultados em 2291.6s. Gravando...
    -> dataset_pequeno_0069.npz gravado.
    -> dataset_pequeno_0070.npz gravado.
    -> dataset_pequeno_0071.npz gravado.
    -> dataset_pequeno_0072.npz gravado.
    -> dataset_pequeno_0073.npz gravado.
    -> dataset_pequeno_0074.npz gravado.
    -> dataset_pequeno_0075.npz gravado.
    -> dataset_pequeno_0076.npz gravado.
    -> dataset_pequeno_0077.npz gravado.
    -> dataset_pequeno_0078.npz gravado.
    -> dataset_pequeno_0079.npz gravado.
    -> dataset_pequeno_0080.npz gravado.
    -> dataset_pequeno_0081.npz gravado.
    -> dataset_pequeno_0082.npz gravado.
    -> dataset_pequeno_0083.npz gravado.
    -> dataset_pequeno_0084.npz gravado.
--- Lote 2/6 (16 NPZs) ---
    Estados únicos a calcular: 62188
    Partições: 360 (~172 estados/partição)
    Spark: 62188 resultados em 2235.8s. Gravando...
    -> dataset_pequeno_008

In [0]:
# ── Auditoria final ───────────────────────────────────────────────────────────

erros = []
total_padrao = 0
total_adaptativo = 0

for path in npzs_originais:
    with np.load(path, allow_pickle=False) as d:
        if 'melhor_jogada' not in d.files or (d['melhor_jogada'] == '').any():
            erros.append(f'{path.name}: melhor_jogada vazia')
            continue
        dm = d['depth_melhor_jogada'].astype(np.int32)
        qtd_tracos = d['qtd_tracos'].astype(np.int32)
        qtd_cad    = d['qtd_cadeias_longas'].astype(np.int32)
        total_cad  = d['total_caixas_cadeias_longas'].astype(np.int32)

    arestas_livres = 31 - qtd_tracos
    prof_min = total_cad + 2 * qtd_cad
    precisa_adaptativo = (arestas_livres > 11) & (prof_min > 11)
    depth_necessaria = np.where(precisa_adaptativo, DEPTH_ADAPTATIVO, DEPTH_PADRAO)

    insuficiente = dm < depth_necessaria
    if insuficiente.any():
        erros.append(f'{path.name}: {insuficiente.sum()} estados com depth < necessária')

    total_padrao      += int((dm == DEPTH_PADRAO).sum())
    total_adaptativo  += int((dm == DEPTH_ADAPTATIVO).sum())

if erros:
    print(f'FALHA: {len(erros)} problema(s):')
    for e in erros[:20]:
        print(f'  {e}')
else:
    print(f'OK: {len(npzs_originais)} NPZs auditados.')
    print(f'  Estados com depth {DEPTH_PADRAO} (padrão)   : {total_padrao:,}')
    print(f'  Estados com depth {DEPTH_ADAPTATIVO} (adaptativo) : {total_adaptativo:,}')
    print('Pronto para fase4_augmentacao_simetria.ipynb.')

OK: 152 NPZs auditados.
  Estados com depth 11 (padrão)   : 747,098
  Estados com depth 20 (adaptativo) : 11,542
Pronto para fase4_augmentacao_simetria.ipynb.
